# 01 - Netflix Catalogue ETL with PySpark

**Goal.** Turn the cleaned Netflix catalogue (one row per title-genre pair) into two analysis-ready tables, each with an explicit grain:

| Table | Grain | Used for |
|---|---|---|
| `titles.csv` | one row per title | KPIs, type mix, freshness, ratings, geography |
| `title_genres.csv` | one row per title x genre | genre analysis only |

**Input.** `data/clean_netflix.csv` is the output of the first-stage cleaning of the public *Netflix Movies and TV Shows* dataset (8,807 titles, up to Sept 2021). If you place the original Kaggle file at `data/raw/netflix_titles.csv`, this notebook cleans it from scratch instead (see `stage_1_from_raw`).

**Design decisions**
1. **Snapshot-anchored age.** Content age is measured against the catalogue snapshot date (the latest `date_added`), not today's date, so every metric is reproducible whenever the notebook is run.
2. **Explicit grain checks.** Title-level counts must never be inflated by the genre explosion.
3. **Repairs with an audit trail.** A handful of source rows have values shifted into the wrong column; they are repaired or flagged, and counted.

In [1]:
import os, json, re
from pathlib import Path
from pyspark.sql import SparkSession, functions as F, Window

spark = (SparkSession.builder.master('local[*]').appName('netflix-content-etl')
         .config('spark.driver.bindAddress', '127.0.0.1').config('spark.driver.host', '127.0.0.1')
         .config('spark.ui.enabled', 'false').config('spark.ui.showConsoleProgress', 'false').config('spark.sql.session.timeZone', 'UTC').getOrCreate())
spark.sparkContext.setLogLevel('ERROR')
DATA = Path('../data'); OUT = DATA / 'processed'; OUT.mkdir(parents=True, exist_ok=True)
print('Spark', spark.version)

Spark 3.5.3


## 1. Load

In [2]:
def stage_1_from_raw(path):
    """Optional: reproduce the first-stage cleaning from the raw Kaggle file (netflix_titles.csv)."""
    raw = spark.read.option('header', True).option('multiLine', True).option('escape', '"').csv(str(path))
    return (raw.dropna(subset=['show_id', 'title']).dropDuplicates(['show_id'])
            .na.fill({'country': 'Unknown', 'director': 'Unknown', 'cast': 'Unknown', 'listed_in': 'Unknown'})
            .withColumn('date_added', F.to_date(F.trim('date_added'), 'MMMM d, yyyy'))
            .withColumn('primary_country', F.trim(F.split('country', ',').getItem(0)))
            .withColumn('genre', F.explode(F.split('listed_in', r',\s*'))))

raw_path = DATA / 'raw' / 'netflix_titles.csv'
if raw_path.exists():
    src = stage_1_from_raw(raw_path); SOURCE = 'raw Kaggle file'
else:
    src = (spark.read.option('header', True).option('multiLine', True).option('escape', '"').csv(str(DATA / 'clean_netflix.csv'))
           .withColumn('date_added', F.to_date('date_added')))
    SOURCE = 'clean_netflix.csv'
src = src.withColumn('release_year', F.col('release_year').cast('double').cast('int')).cache()
n_rows, n_titles = src.count(), src.select('show_id').distinct().count()
print(f'Source: {SOURCE} | rows: {n_rows:,} | distinct titles: {n_titles:,} | rows per title: {n_rows / n_titles:.2f}')
src.printSchema()

Source: clean_netflix.csv | rows: 19,304 | distinct titles: 8,807 | rows per title: 2.19
root
 |-- show_id: string (nullable = true)
 |-- type: string (nullable = true)
 |-- title: string (nullable = true)
 |-- director: string (nullable = true)
 |-- cast: string (nullable = true)
 |-- country: string (nullable = true)
 |-- date_added: date (nullable = true)
 |-- release_year: integer (nullable = true)
 |-- rating: string (nullable = true)
 |-- duration: string (nullable = true)
 |-- listed_in: string (nullable = true)
 |-- description: string (nullable = true)
 |-- content_age: string (nullable = true)
 |-- primary_country: string (nullable = true)
 |-- genre: string (nullable = true)



**Observation:** 19,304 rows describe 8,807 distinct titles, an average of 2.19 rows per title, because each title is repeated once per genre tag. Any `COUNT(*)` on this table overstates the catalogue by more than 2x. That is why the pipeline produces a separate title-level table.

## 2. Grain and Quality Audit

In [3]:
genres_per_title = src.groupBy('show_id').agg(F.countDistinct('genre').alias('g'))
genres_per_title.groupBy('g').count().orderBy('g').show()
titles_raw = src.dropDuplicates(['show_id'])
nulls = titles_raw.select([F.sum(F.col(c).isNull().cast('int')).alias(c) for c in ['date_added', 'release_year', 'rating', 'duration', 'primary_country']])
nulls.show()
unknown = titles_raw.select([F.sum((F.col(c) == 'Unknown').cast('int')).alias(c) for c in ['director', 'cast', 'country']])
unknown.show()
VALID_RATINGS = ['TV-MA', 'TV-14', 'TV-PG', 'R', 'PG-13', 'TV-Y7', 'TV-Y', 'PG', 'TV-G', 'NR', 'G', 'TV-Y7-FV', 'NC-17', 'UR', 'Not Rated']
bad = titles_raw.filter(~F.col('rating').isin(VALID_RATINGS) | F.col('rating').isNull())
print('Titles with an invalid or missing rating:', bad.count())
bad.select('show_id', 'title', 'rating', 'duration').show(truncate=40)

+---+-----+
|  g|count|
+---+-----+
|  1| 2032|
|  2| 3053|
|  3| 3722|
+---+-----+



+----------+------------+------+--------+---------------+
|date_added|release_year|rating|duration|primary_country|
+----------+------------+------+--------+---------------+
|        30|          20|     0|       4|              2|
+----------+------------+------+--------+---------------+



+--------+----+-------+
|director|cast|country|
+--------+----+-------+
|    2635| 825|    831|
+--------+----+-------+



Titles with an invalid or missing rating: 22


+-------+---------------------------------------+-----------------------+-----------------+
|show_id|                                  title|                 rating|         duration|
+-------+---------------------------------------+-----------------------+-----------------+
|  s1252|                     Dances with Wolves|          Maury Chaykin|    Jimmy Herman"|
|  s1412|            What Would Sophia Loren Do?|                   2021|            TV-14|
|  s1416|                    An Imperfect Murder|                   2017|                R|
|  s1549|                          The Challenge|                   2006|            TV-14|
|   s171|                         Rhyme & Reason|                   E-40|          MC Eiht|
|  s1769|                    The Next Karate Kid|       November 1, 2020|             1994|
|   s199|   King of Boys: The Return of the King| Keppy Ekpenyong Bassey|     Bimbo Manuel|
|  s2138|                           An Easy Girl|                   2019|       

**Observation:** Each title carries one to three genre tags (2,032 / 3,053 / 3,722 titles). Director (2,635), cast (825) and country (831) are often recorded as 'Unknown'. **22 titles have values shifted into the rating column**: cast names, years, dates, or durations such as '74 min' for three Louis C.K. specials. These come from unescaped quotes and commas in the original CSV.

## 3. Repairs and Features

In [4]:
snapshot = titles_raw.agg(F.max('date_added')).first()[0]
SNAP_YEAR = snapshot.year
print('Catalogue snapshot date (latest date_added):', snapshot)

def build_title_features(df):
    return (df
        # (a) rating holds a duration such as "74 min" (the value shifted one column left in the source)
        .withColumn('duration', F.when(F.col('rating').rlike(r'^\d+ min$') & F.col('duration').isNull(), F.col('rating')).otherwise(F.col('duration')))
        .withColumn('rating_repaired', ~F.col('rating').isin(VALID_RATINGS) & F.col('rating').isNotNull())
        .withColumn('rating', F.when(F.col('rating').isin(VALID_RATINGS), F.col('rating'))
                               .when(F.col('rating').isin('NR', 'UR'), 'Not Rated').otherwise('Unknown'))
        .withColumn('rating', F.when(F.col('rating').isin('NR', 'UR'), 'Not Rated').otherwise(F.col('rating')))
        .withColumn('audience', F.when(F.col('rating').isin('TV-Y', 'TV-Y7', 'TV-Y7-FV', 'TV-G', 'G', 'PG', 'TV-PG'), 'Kids & family')
                                  .when(F.col('rating').isin('TV-14', 'PG-13'), 'Teens')
                                  .when(F.col('rating').isin('TV-MA', 'R', 'NC-17'), 'Adults').otherwise('Unrated'))
        .withColumn('duration_value', F.regexp_extract('duration', r'(\d+)', 1).cast('int'))
        .withColumn('duration_unit', F.when(F.col('duration').contains('min'), 'min').when(F.col('duration').contains('Season'), 'seasons'))
        .withColumn('n_countries', F.when(F.col('country') == 'Unknown', 0).otherwise(F.size(F.split('country', ','))))
        .withColumn('year_added', F.year('date_added'))
        .withColumn('content_age', F.lit(SNAP_YEAR) - F.col('release_year'))          # snapshot-anchored
        .withColumn('content_age_group', F.when(F.col('content_age') <= 2, 'Very recent (0-2y)').when(F.col('content_age') <= 5, 'Recent (3-5y)')
                                           .when(F.col('content_age') <= 10, 'Mature (6-10y)').when(F.col('content_age') <= 20, 'Older (11-20y)')
                                           .otherwise('Legacy (20y+)'))
        .withColumn('release_decade', (F.floor(F.col('release_year') / 10) * 10).cast('int'))
        .withColumn('is_recent', (F.col('content_age') <= 5).cast('int'))
        .withColumn('add_lag_years', F.col('year_added') - F.col('release_year'))
        .withColumn('is_original_window', (F.col('add_lag_years') <= 1).cast('int')))      # added within a year of release

titles = (build_title_features(titles_raw)
          .join(genres_per_title.withColumnRenamed('g', 'genre_count'), 'show_id')
          .select('show_id', 'title', 'type', 'primary_country', 'n_countries', 'release_year', 'date_added', 'year_added',
                  'rating', 'audience', 'rating_repaired', 'duration_value', 'duration_unit', 'content_age', 'content_age_group',
                  'release_decade', 'is_recent', 'add_lag_years', 'is_original_window', 'genre_count')).cache()
print('Repaired / flagged rating values:', titles.filter('rating_repaired').count())
titles.groupBy('rating').count().orderBy(F.desc('count')).show(20)

Catalogue snapshot date (latest date_added): 2021-09-25


Repaired / flagged rating values: 22


+---------+-----+
|   rating|count|
+---------+-----+
|    TV-MA| 3195|
|    TV-14| 2158|
|    TV-PG|  862|
|        R|  796|
|    PG-13|  489|
|    TV-Y7|  334|
|     TV-Y|  307|
|       PG|  286|
|     TV-G|  220|
|Not Rated|   88|
|        G|   41|
|  Unknown|   22|
| TV-Y7-FV|    6|
|    NC-17|    3|
+---------+-----+



**Observation:** The latest `date_added` is 25 September 2021, which is used as the snapshot date. The three shifted durations are recovered into `duration`. All 22 broken ratings are set to 'Unknown' and flagged with `rating_repaired`, so they can be excluded from rating analysis without losing the titles. 'NR' and 'UR' are merged into 'Not Rated' (88 titles).

In [5]:
# Why snapshot anchoring matters: the same release year gives a different "age" depending on when you run the code
from datetime import date
compare = titles.select(
    F.avg((F.lit(SNAP_YEAR) - F.col('release_year') <= 5).cast('int')).alias('recent_share_snapshot'),
    F.avg((F.lit(date.today().year) - F.col('release_year') <= 5).cast('int')).alias('recent_share_runtime')).first()
print(f"Share of titles 'recent' (<= 5 years old) - anchored to {SNAP_YEAR}: {compare['recent_share_snapshot']:.1%} | "
      f"anchored to today ({date.today().year}): {compare['recent_share_runtime']:.1%}")

Share of titles 'recent' (<= 5 years old) - anchored to 2021: 64.2% | anchored to today (2026): 6.7%


**Observation:** This is the most consequential fix in the pipeline. Anchored to the 2021 snapshot, **64.2%** of titles are 'recent' (released within five years). Computing age from today's date, as the original pipeline did, gives **6.7%** when run in 2026, and the figure keeps falling every year the dashboard is reopened. Every freshness KPI built on a runtime clock is silently wrong.

## 4. Genre Table and Validation

In [6]:
title_genres = (src.select('show_id', F.trim('genre').alias('genre')).filter(F.col('genre') != 'Unknown').dropDuplicates()
                .join(titles.select('show_id', 'type', 'year_added', 'release_year', 'primary_country'), 'show_id'))
checks = {
    'titles: one row per show_id': titles.count() == titles.select('show_id').distinct().count(),
    'titles: count equals source distinct titles': titles.count() == n_titles,
    'title_genres: no duplicate (show_id, genre)': title_genres.count() == title_genres.dropDuplicates(['show_id', 'genre']).count(),
    'every genre row maps to a title': title_genres.join(titles, 'show_id', 'left_anti').count() == 0,
    'type is Movie or TV Show': titles.filter(~F.col('type').isin('Movie', 'TV Show')).count() == 0,
}
for k, v in checks.items(): print(('PASS ' if v else 'FAIL ') + k)
assert all(checks.values())
print(f'titles: {titles.count():,} rows | title_genres: {title_genres.count():,} rows | distinct genres: {title_genres.select("genre").distinct().count()}')

PASS titles: one row per show_id
PASS titles: count equals source distinct titles
PASS title_genres: no duplicate (show_id, genre)
PASS every genre row maps to a title
PASS type is Movie or TV Show


titles: 8,807 rows | title_genres: 19,303 rows | distinct genres: 64


**Observation:** All five grain and integrity checks pass: one row per title, no lost titles, no duplicate title-genre pairs, no orphan genre rows, and only valid content types. The genre table has 19,303 rows across 64 genres (one 'Unknown' genre row is dropped).

## 5. Export

In [7]:
titles_pd = titles.orderBy('show_id').toPandas()
genres_pd = title_genres.orderBy('show_id', 'genre').toPandas()
INT_COLS = ['n_countries', 'release_year', 'year_added', 'duration_value', 'content_age', 'release_decade', 'is_recent',
            'add_lag_years', 'is_original_window', 'genre_count']
titles_pd[INT_COLS] = titles_pd[INT_COLS].astype('Int64')            # nullable integers: no "2020.0" in the CSV
genres_pd[['year_added', 'release_year']] = genres_pd[['year_added', 'release_year']].astype('Int64')
titles_pd.to_csv(OUT / 'titles.csv', index=False)
genres_pd.to_csv(OUT / 'title_genres.csv', index=False)
kpis = {'snapshot_date': str(snapshot), 'titles': int(len(titles_pd)), 'movies': int((titles_pd.type == 'Movie').sum()),
        'tv_shows': int((titles_pd.type == 'TV Show').sum()), 'countries': int(titles_pd.loc[titles_pd.primary_country != 'Unknown', 'primary_country'].nunique()),
        'genres': int(genres_pd.genre.nunique()), 'median_content_age': float(titles_pd.content_age.median()),
        'recent_share': round(float(titles_pd.is_recent.mean()), 4)}
json.dump(kpis, open(OUT / 'kpis.json', 'w'), indent=2)
print(json.dumps(kpis, indent=2))
spark.stop()

{
  "snapshot_date": "2021-09-25",
  "titles": 8807,
  "movies": 6131,
  "tv_shows": 2676,
  "countries": 104,
  "genres": 64,
  "median_content_age": 4.0,
  "recent_share": 0.6422
}


**Observation:** Exported `titles.csv` (8,807 rows), `title_genres.csv` (19,303 rows) and `kpis.json`. The catalogue has 6,131 movies and 2,676 TV shows from 104 primary production countries. The median title was 4 years old at the snapshot date. Integer columns are written as nullable integers so the files load cleanly into MySQL INT columns.

## ETL summary

| Step | Result |
|---|---|
| Source grain | 19,304 title-genre rows for 8,807 titles (2.19x inflation) |
| Repairs | 22 shifted ratings flagged; 3 durations recovered; NR/UR merged into Not Rated |
| Snapshot anchoring | 'Recent' share 64.2% (snapshot) vs 6.7% (runtime clock, 2026) |
| Outputs | `titles.csv` (1 row per title), `title_genres.csv` (1 row per title x genre), `kpis.json` |
| Integrity checks | 5 of 5 pass |

Next: `02_content_strategy_analysis.ipynb` analyses these tables, and `app/app.py` serves them as a dashboard.